## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [3]:
import glob
import pyarrow.parquet as pq
import pandas as pd

from Challenge.paths import XGBOOST_DATAFRAMES

Running on local — storage at: /home/luigi/RecSys


## **Merge fold dataframes**

In [4]:
INPUT_DIR = os.path.join(XGBOOST_DATAFRAMES, "ready_for_xgboost")
input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "clean_part_*.parquet")))
output_file = os.path.join(XGBOOST_DATAFRAMES, "training_data.parquet")

print(f"Merging {len(input_files)} files into '{output_file}'...")

Merging 10 files into '/home/luigi/RecSys/xg_boost_data/dataframes/training_data.parquet'...


In [5]:
print(f"Merging {len(input_files)} files into '{output_file}'...")

# 2. Get Schema from the first file
schema = pq.ParquetFile(input_files[0]).schema_arrow

# 3. Stream content to the new file
with pq.ParquetWriter(output_file, schema, compression='snappy') as writer:
    for i, f in enumerate(input_files):
        # Read file into memory (small chunk)
        table = pq.read_table(f)
        
        # Write to master file
        writer.write_table(table)
        
        print(f"[{i+1}/{len(input_files)}] Appended {os.path.basename(f)}")

print(f"\nDone! Master file created: {output_file}")

Merging 10 files into '/home/luigi/RecSys/xg_boost_data/dataframes/training_data.parquet'...
[1/10] Appended clean_part_0.parquet
[2/10] Appended clean_part_1.parquet
[3/10] Appended clean_part_2.parquet
[4/10] Appended clean_part_3.parquet
[5/10] Appended clean_part_4.parquet
[6/10] Appended clean_part_5.parquet
[7/10] Appended clean_part_6.parquet
[8/10] Appended clean_part_7.parquet
[9/10] Appended clean_part_8.parquet
[10/10] Appended clean_part_9.parquet

Done! Master file created: /home/luigi/RecSys/xg_boost_data/dataframes/training_data.parquet
